# Stage 4 — Rolling-origin cross-validation

Multiple time-based train/test cutoffs, mimicking how the model would actually be retrained and re-forecast over time.

In [1]:
import pandas as pd
from pathlib import Path

BASE_DIR = Path("/Users/asadkhan.pathan/Documents/New project/m5 forecasting ")
df = pd.read_parquet(BASE_DIR / "data" / "processed" / "sales_long.parquet")
df = df.sort_values(["id", "date"]).reset_index(drop=True)

HORIZON = 28
N_FOLDS = 3
last_date = df["date"].max()

# Fold 0 = most recent (cutoff closest to end of data), fold 2 = oldest.
# Each cutoff is HORIZON days further back than the previous one, so the
# 3 test windows sit back-to-back with zero overlap.
fold_cutoffs = [last_date - pd.Timedelta(days=HORIZON * (i + 1)) for i in range(N_FOLDS)]

for i, cutoff in enumerate(fold_cutoffs):
    test_start = cutoff + pd.Timedelta(days=1)
    test_end = cutoff + pd.Timedelta(days=HORIZON)
    print(f"Fold {i}: train ends {cutoff.date()}  |  test = {test_start.date()} to {test_end.date()}")


Fold 0: train ends 2016-04-24  |  test = 2016-04-25 to 2016-05-22
Fold 1: train ends 2016-03-27  |  test = 2016-03-28 to 2016-04-24
Fold 2: train ends 2016-02-28  |  test = 2016-02-29 to 2016-03-27


## Step 2: Naive aur seasonal-naive, har fold pe

Stage 2 ka baseline logic ek function mein daala, taaki teeno folds pe
reuse ho sake. Maksad: dekhna ki score fold-to-fold kitna stable hai —
ye hi rolling-origin CV ka pura point hai, sirf ek number pe bharosa
nahi karna.


In [4]:
import numpy as np

def rmse(actual, pred):
    return np.sqrt(np.mean((actual - pred) ** 2))

def mae(actual, pred):
    return np.mean(np.abs(actual - pred))

def evaluate_fold(df, cutoff, horizon=28):
    train = df[df["date"] <= cutoff]
    test = df[(df["date"] > cutoff) & (df["date"] <= cutoff + pd.Timedelta(days=horizon))].copy()

    # Naive: last known training value, held flat.
    last_actual = train.groupby("id", observed=True)["sales"].last()
    test = test.merge(last_actual.rename("naive_pred"), on="id", how="left")

    # Seasonal-naive: tile the last 7 training days across the horizon.
    last7 = train.groupby("id", observed=True).tail(7).copy()
    last7["pos"] = last7.groupby("id", observed=True).cumcount()
    last7_wide = last7.pivot(index="id", columns="pos", values="sales")

    test = test.sort_values(["id", "date"])
    test["day_offset"] = test.groupby("id", observed=True).cumcount()
    test["pos"] = test["day_offset"] % 7
    last7_long = last7_wide.reset_index().melt(id_vars="id", var_name="pos", value_name="seasonal_naive_pred")
    test = test.merge(last7_long, on=["id", "pos"], how="left")

    return {
        "naive_RMSE": rmse(test["sales"], test["naive_pred"]),
        "naive_MAE": mae(test["sales"], test["naive_pred"]),
        "seasonal_naive_RMSE": rmse(test["sales"], test["seasonal_naive_pred"]),
        "seasonal_naive_MAE": mae(test["sales"], test["seasonal_naive_pred"]),
    }

fold_results = []
for i, cutoff in enumerate(fold_cutoffs):
    result = evaluate_fold(df, cutoff)
    result["fold"] = i
    fold_results.append(result)

fold_results_df = pd.DataFrame(fold_results).set_index("fold")
print(fold_results_df)


      naive_RMSE  naive_MAE  seasonal_naive_RMSE  seasonal_naive_MAE
fold                                                                
0       2.893557   1.373017             2.676895            1.243972
1       3.300666   1.350344             2.759459            1.232355
2       3.127850   1.392495             2.746322            1.210618
